# 02 · Cleaning the data, step by step

**Goal:** see exactly how each problem found in notebook 01 gets fixed. The real pipeline (`src/clean.py`) does all of this automatically; here we do it slowly, on a few rows, so you can watch.

**Folder reminder:**
- `data/raw/`: the original files, never changed
- `data/interim/`: each table cleaned, before joining
- `data/processed/`: everything joined, ready for models

In [1]:
# --- setup: run this cell first ---
import sys
sys.path.insert(0, "../src")          # so we can use the project's own helpers

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils import path                # path("raw", "file.csv") -> full path to data/raw/file.csv

plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.width", 120)

In [2]:
raw = pd.read_csv(path("raw", "traffic_sensor_log.csv"))
print(raw.shape)

(178468, 17)


## Fix 1 · Remove duplicate rows

In [3]:
before = len(raw)
deduped = raw.drop_duplicates()
print("removed", before - len(deduped), "duplicate rows")

removed 1767 duplicate rows


## Fix 2 · Repair negative volumes by copying `vehicle_count`

Notebook 01 showed that `vehicle_count` always equals `traffic_volume` on good rows. So a broken volume can be copied back exactly.

In [4]:
bad = deduped["traffic_volume"] < 0
print("before:"); print(deduped.loc[bad, ["road_id", "traffic_volume", "vehicle_count"]].head(3))

fixed = deduped.copy()                                  # work on a copy, keep the original
fixed.loc[bad, "traffic_volume"] = fixed.loc[bad, "vehicle_count"]

print("\nafter:"); print(fixed.loc[bad, ["road_id", "traffic_volume", "vehicle_count"]].head(3))

before:
     road_id  traffic_volume  vehicle_count
676   NL-009            -6.0            247
1065  NL-014           -33.0            119
1397  NL-007           -38.0            258



after:
     road_id  traffic_volume  vehicle_count
676   NL-009           247.0            247
1065  NL-014           119.0            119
1397  NL-007           258.0            258


## Fix 3 · Repair impossible speeds from travel time

Travel time is computed as `60 × road length ÷ speed`. Flip it around: `speed = 60 × length ÷ travel time`. First we find each road's length from the good rows.

In [5]:
good = fixed[fixed["avg_speed"].between(1, 200)]
length = (good["travel_time"] * good["avg_speed"] / 60).groupby(good["road_id"]).median()
print("road lengths in km (first 5):"); print(length.head().round(2))

too_fast = fixed["avg_speed"] > 200
example = fixed[too_fast].iloc[0]
true_speed = 60 * length[example["road_id"]] / example["travel_time"]
print(f"\n{example['road_id']} says {example['avg_speed']} km/h -> real speed {true_speed:.1f} km/h")

road lengths in km (first 5):
road_id
NL-001    1.99
NL-002    0.86
NL-003    2.02
NL-004    0.83
NL-005    1.06
dtype: float64

NL-002 says 317.0 km/h -> real speed 50.6 km/h


## Fix 4 · Missing *rows*, not just missing values

Each road should have one row every 30 minutes, which is 48 per day. Let's count how many are actually there.

In [6]:
expected = raw["road_id"].nunique() * raw["date"].nunique() * 48
actual = len(fixed.drop_duplicates(["road_id", "date", "time"]))
print("expected rows:", expected)
print("actual rows:  ", actual)
print("missing windows:", expected - actual)

expected rows: 181200
actual rows:   176701
missing windows: 4499


**Why this matters:** later, "the previous row" is used to mean "30 minutes ago" (a *lag* feature). If a row is missing, the previous row might be an hour earlier, and nothing would warn us.

So the pipeline **rebuilds every missing window**, fills its numbers (short gaps: a straight line between neighbours; long gaps: that road's usual value at that hour), and marks it `is_imputed = 1` so it's **never used as an answer to learn from**.

## Fix 5 · One spelling per weather type

In [7]:
weather = pd.read_csv(path("raw", "weather_observations.csv"))
mapping = {"clear": "Clear", "cloudy": "Cloudy", "overcast": "Overcast",
           "rain": "Rain", "rainy": "Rain", "fog": "Fog", "foggy": "Fog"}

clean_labels = weather["weather_condition"].str.strip().str.lower().map(mapping)
print("before:", weather["weather_condition"].nunique(), "spellings")
print("after: ", clean_labels.nunique(), "labels ->", sorted(clean_labels.unique()))

before: 13 spellings
after:  5 labels -> ['Clear', 'Cloudy', 'Fog', 'Overcast', 'Rain']


## Fix 6 · Two date formats

Traffic writes dates as `2025-03-04`, weather as `04/03/2025`. Read the weather one the wrong way round and 4 March becomes 3 April, and the weather would be joined to the wrong day.

In [8]:
print(pd.to_datetime("04/03/2025", format="%d/%m/%Y"))   # correct: day first -> 4 March
print(pd.to_datetime("04/03/2025", format="%m/%d/%Y"))   # wrong: month first -> 3 April

2025-03-04 00:00:00
2025-04-03 00:00:00


## The result

The full pipeline does all of the above, joins the three tables, and checks the result. Here is its report, generated by the code with every number counted, not typed:

In [9]:
print(path("reports", "data_quality.md").read_text())

# FlowCast - Data Quality Report

Generated by the pipeline from `traffic_sensor_log.csv` (sha256 8f793f3643c8). Every number below is counted by the code, not typed in.

## Defects found and how each was resolved

| Defect (PRD 8.3) | Found | Resolution |
|---|---|---|
| Duplicate log rows | 1767 exact, 0 key-level | Dropped; for key duplicates the most complete row was kept |
| Negative or missing traffic_volume | 4587 | Recovered exactly from vehicle_count (equal on every valid row) |
| avg_speed above 200 km/h | 4579 | Recovered exactly as 60 x segment length / travel_time |
| Occupancy above 100% | 229 | Capped at 100 |
| Statistically extreme readings (robust z > 4.0 vs the same segment and time-of-day slot) | volume 26, speed 463, occupancy 225 | Flagged and kept: after the rule fixes these are mostly real events and incidents |
| Missing sensor windows (absent rows) | 4499 | Rebuilt on the full segment x 30-min grid, flagged is_imputed = 1 |
| Values still missing after recover

Next: **03_modelling.ipynb**, where models learn from this clean table.